# 02. Data preparation and availability timing

**The question.** How do raw prices, yields, and mortgage statistics become clean daily and monthly panels in which every value is dated by when it could actually have been known?

**Why a portfolio manager cares.** A hedge decided at a month-end close can only use information released by that close. Getting the timing wrong makes any backtest look better than real trading.

**Learning objectives**

1. Build one trading calendar on which the MBS return and the yield change always cover the same days.
2. Rebuild MBB total returns from closes and distributions, and check them against the adjusted close.
3. Price the synthetic 10-year hedge bond and verify one day by hand.
4. Construct the refinancing incentive and the share of borrowers in the money, joined on release dates.
5. Record a data-quality report and the data dictionary.

| | |
|---|---|
| Inputs | raw files and manifest from notebook 01 |
| Outputs | `data/processed/daily_panel.parquet`, `results/state/state_info.json`, `results/data_quality/quality_report.json`, `results/state/data_dictionary.csv` |
| Run after | 01 |
| Approximate runtime | under a minute |

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import pipeline as pl
from mbshedge import prepare as pr
from mbshedge import instruments as ins
from mbshedge.sources import RawStore, load_raw_tables

tables = load_raw_tables(RawStore(paths.raw), cfg)
print("FRED series:", sorted(tables["fred"]), "| prices:", sorted(tables["prices"]), "| NMDB rows:", len(tables["nmdb"]))

## 1. The trading calendar and daily returns

**Calendar.** We keep a date only if MBB has a close *and* the 10-year yield is published. On bond-market holidays such as Columbus Day the ETF still trades but no yield is published. Its return on that day is compounded into the next calendar date, so the MBS return and the yield change always span the same interval.

**Total return.** For each MBB trading day $d$,

$$
\text{TR}_d = \frac{P_d + \text{Div}_d}{P_{d-1}} - 1 ,
$$

where $P_d$ is the close (split-adjusted, not distribution-adjusted) and $\text{Div}_d$ is the cash distribution with ex-date $d$. Distributions enter once. We never add them to an adjusted price.

**Excess return.** Over an interval of $n$ calendar days, the bill accrual is $r_f = y_{3m}/100 \times n/365$, using the 3-month yield at the start of the interval. MBS and hedge returns are both measured in excess of $r_f$.

**Units.** Yields are in percent, yield changes `dy10` in decimals (0.0001 = 1 bp), returns in decimals.

In [ ]:
prep = pl.prepare(tables, cfg, EXP)
daily = prep["daily"]
quality = prep["quality"]
print(f"Calendar: {len(daily)} trading dates from {daily.index.min().date()} to {daily.index.max().date()}")
print(json.dumps(quality["calendar"], indent=1))
daily[["tr_mbb", "rf", "r_mbb_ex", "y10", "dy10", "r_h10_ex", "dur_h10", "pmms", "stock_rate", "incentive", "g", "incentive_source"]].tail()

### Check: our total return against Yahoo's adjusted close

If distributions were missed, the two would differ by about 0.3% on each ex-date. If they were double counted, our return would exceed the adjusted-close return by the distribution yield. Small differences are expected: Yahoo reinvests at the ex-date close, while our formula reinvests at the prior close, which adds up to about 1 bp a year. The panel builder enforces a limit of 10 bp a year: above it, the run stops for MBB, and VMBS, PMBS, or PIMCO's GNMA fund (PDMIX) is left out of the panel. A provider without an adjusted close (a local file, for example) cannot be checked this way.

In [ ]:
for k in [k for k in quality if k.startswith("prices_")]:
    q = quality[k]
    print(f"{k}: {q['rows']} rows, {q['n_distributions']} distributions, duplicates dropped {q['duplicates_dropped']}, "
          f"max |TR - adjusted-close return| = {q.get('adjclose_check_max_abs_diff_bp', float('nan')):.3f} bp, "
          f"days above 1 bp: {q.get('adjclose_check_days_over_1bp', 'n/a')}, "
          f"cumulative gap {q.get('adjclose_check_cum_diff_bp_per_year', float('nan')):.3f} bp per year")
    if q.get("adjclose_check_dates_over_1bp"):
        print(f"   dates with a gap above 1 bp (first 10): {', '.join(q['adjclose_check_dates_over_1bp'])}")
    if "adjclose_check_cum_diff_bp_per_year" not in q:
        print("   no adjusted close from this provider, so the distribution check could not run")
    if q.get("excluded"):
        print(f"   EXCLUDED: {q['excluded']}")

## 2. The hedge instrument, worked by hand

The hedge is a synthetic 10-year par Treasury. At each close we hold a new 10-year bond whose coupon equals that day's yield, so it is worth exactly 100. One trading interval later (Δt = calendar days ÷ 365.25 years) the bond has aged slightly and is revalued at the new yield:

$$
P_t = \sum_{k=1}^{20} \frac{CF_k}{(1 + y_t/2)^{\,2(k/2 - \Delta t)}}, \qquad
CF_k = 100\cdot\frac{y_{t-1}}{2} \;(+100 \text{ at } k = 20), \qquad
r^{H}_t = \frac{P_t}{100} - 1 .
$$

Its modified duration at the close is that of a new par bond, $D_H(y) = \frac{1}{y}\left[1 - (1 + y/2)^{-20}\right]$.

This is a research approximation of a rolling 10-year position. It is not a futures contract: there is no basis, delivery option, or roll. The flat-curve shortcut discounts the aged bond at the 10-year yield, which is accurate for a few days of ageing.

In [ ]:
i = len(daily) // 2
row, prev = daily.iloc[i], daily.iloc[i - 1]
y0, y1, days = prev["y10"] / 100, row["y10"] / 100, row["days"]
dt = days / 365.25
times = np.arange(1, 21) / 2 - dt
cf = np.full(20, 100 * y0 / 2); cf[-1] += 100
price_by_hand = (cf * (1 + y1 / 2) ** (-2 * times)).sum()
print(f"Date {daily.index[i].date()}: yield {100*y0:.2f}% -> {100*y1:.2f}% over {days:.0f} day(s)")
print(f"  price by hand      {price_by_hand:.6f}  -> return {price_by_hand/100 - 1:+.6%}")
print(f"  pipeline return    {row['r_h10']:+.6%}")
print(f"  approximation -D*dy + y*dt = {-ins.par_bond_modified_duration(y0, 10)[0]*(y1-y0) + y0*dt:+.6%}")
assert abs(price_by_hand / 100 - 1 - row["r_h10"]) < 1e-12

## 3. The refinancing state, as it was known at each close

Three ingredients, each joined on the date it became available:

* **PMMS** (current origination rate for a conforming 30-year loan): weekly, usable from the Thursday release.
* **Stock note rate** $S$: the average note rate on outstanding agency-eligible mortgages, the UPB-weighted average of NMDB's *Enterprise Acquisitions* and *Government / Non-Conventional* markets. A quarter becomes usable at the end of the **following** quarter (2026Q2 was released on 2026-09-30). NMDB was first published long after 2013, so this lag is an availability approximation. The information itself (the coupon of the outstanding agency stock) was available to market participants from agency pool disclosures.
* **Before NMDB** (until mid-2013) the stock rate comes from an exponentially weighted average of past PMMS rates, with its half-life and level fitted to the NMDB quarters available by December 2014. That fit uses 2013–2014 information, so every value before mid-2013 is labeled `proxy_retrospective`. It is used for development and teaching, never as historical trading evidence.

The incentive is $I = S - \text{PMMS}$ in percentage points. The share of borrowers in the money assumes note rates spread around $S$ with standard deviation $\sigma$:

$$
g = \Pr(\text{note rate} \ge \text{PMMS} + \theta) = \Phi\!\left(\frac{I - \theta}{\sigma}\right), \qquad \theta = 0.5 \text{ pp}.
$$

$\sigma$ is estimated from NMDB's rate buckets available by the end of development. All three are *proxies* for the incentive of MBB's own borrowers. They are not MBB's weighted coupon, which iShares reports only for the current date.

In [ ]:
info = prep["state_info"]
proxy = info["proxy"]
print("Markets:", info["markets"], "| NMDB lag (quarters):", info["lag_quarters"])
print(f"Proxy calibration ({proxy.get('status')}): half-life {proxy.get('halflife_years')} years, "
      f"offset {proxy.get('offset', float('nan')):+.3f} pp, RMSE {proxy.get('rmse_pp', float('nan')):.3f} pp "
      f"on {proxy.get('n_quarters')} development quarters")
print(f"sigma = {info['sigma_pp']} pp ({info['sigma_source']}), theta = {info['theta_pp']} pp")
pd.DataFrame(proxy.get("grid", []))

In [ ]:
disp = info.get("bucket_dispersion")
if disp is not None and len(disp):
    disp = disp.copy()
    disp["available_from"] = pr.availability_dates(disp["quarter_end"], info["lag_quarters"])
    dev_disp = disp[disp["available_from"] <= pd.Timestamp(EXP["sample"]["development_end"]) + pd.offsets.MonthEnd(0)]
    print("Note-rate dispersion implied by NMDB rate buckets (development quarters only):")
    display(dev_disp.round({"mean_from_buckets": 3, "sd_from_buckets": 3}))

### What was known when: the stock note rate by quarter-end versus by availability

The step line is what a decision-maker could use on each date. It lags the quarter-end values (dots) by the release delay.

In [ ]:
from mbshedge.prepare import nmdb_stock_rate
sq = nmdb_stock_rate(tables["nmdb"], cfg["data"]["nmdb"]["markets_core"])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(daily.index, daily["pmms"], color="#8c8c8c", lw=1.2, label="PMMS 30-year rate (by release)")
nm = daily["incentive_source"] == "nmdb"
ax.plot(daily.index[nm], daily.loc[nm, "stock_rate"], color=pt.INK, lw=1.6, drawstyle="steps-post", label="Stock note rate as available (NMDB)")
px = daily["incentive_source"] == "proxy_retrospective"
ax.plot(daily.index[px], daily.loc[px, "stock_rate"], color=pt.INK, lw=1.4, ls="--", label="Stock note rate, retrospective proxy")
ax.scatter(sq["quarter_end"], sq["stock_rate"], s=14, color="#2a78d6", zorder=3, label="NMDB value at its quarter end")
ax.set_ylabel("percent"); ax.set_title("Inputs to the refinancing incentive (full sample, input data)")
ax.legend(loc="upper right")
written = pt.save_figure(fig, "02_incentive_inputs", paths, data=daily[["pmms", "stock_rate", "incentive_source"]], mode=MODE)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
axes[0].plot(daily.index, daily["incentive"], color=pt.INK, lw=1.2)
axes[0].axhline(0, color=pt.INK_2, lw=0.8)
axes[0].axhline(EXP["incentive"]["refi_threshold_pp"], color=pt.INK_2, lw=0.8, ls=":")
axes[0].set_ylabel("percentage points"); axes[0].set_title("Refinancing incentive I = stock note rate - PMMS")
axes[1].plot(daily.index, daily["g"], color="#1baf7a", lw=1.4)
axes[1].set_ylabel("share of borrowers"); axes[1].set_ylim(0, 1)
axes[1].set_title("Estimated share of borrowers in the money, g = Phi((I - theta) / sigma)")
for ax in axes:
    ax.axvspan(daily.index.min(), daily.index[daily["incentive_source"] == "nmdb"].min(), color=pt.EPISODE_FILL, lw=0)
axes[0].text(daily.index.min(), 0.95, " shaded: proxy (retrospective)", transform=axes[0].get_xaxis_transform(), va="top", fontsize=8, color=pt.INK_2)
pt.save_figure(fig, "02_incentive_and_share", paths, data=daily[["incentive", "g", "incentive_source"]], mode=MODE)
plt.show()

## 4. Timing checks

These assertions must hold for every row. They are the data-level version of the look-ahead test in notebook 06.

In [ ]:
checks = {}
checks["PMMS release date never after the row date"] = bool((daily["pmms_release_date"] <= daily.index).all())
nmr = daily["stock_rate_available_from"].notna()
checks["NMDB availability never after the row date"] = bool((daily.loc[nmr, "stock_rate_available_from"] <= daily.index[nmr]).all())
checks["NMDB quarter end at least one quarter before use"] = bool(((daily.index[nmr] - daily.loc[nmr, "stock_rate_quarter_end"]).dt.days >= 89).all())
checks["no duplicate trading dates"] = bool(daily.index.is_unique)
checks["calendar strictly increasing"] = bool(daily.index.is_monotonic_increasing)
checks["no missing MBB excess returns"] = bool(daily["r_mbb_ex"].notna().all())
checks["no missing 10-year hedge returns"] = bool(daily["r_h10_ex"].notna().all())
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
assert all(checks.values()), "A timing or completeness check failed"

## 5. Save the panels, the state choices, the quality report, and the data dictionary

In [ ]:
(paths.processed).mkdir(parents=True, exist_ok=True)
daily.to_parquet(paths.processed / "daily_panel.parquet")
state_dir = paths.results / "state"; state_dir.mkdir(parents=True, exist_ok=True)
info_out = {k: v for k, v in info.items() if k != "bucket_dispersion"}
(state_dir / "state_info.json").write_text(json.dumps(info_out, indent=2, default=str))
if disp is not None and len(disp):
    disp.to_csv(state_dir / "nmdb_bucket_dispersion.csv", index=False)
qdir = paths.results / "data_quality"; qdir.mkdir(parents=True, exist_ok=True)
(qdir / "quality_report.json").write_text(json.dumps({**quality, "timing_checks": checks}, indent=2, default=str))
pr.dictionary_frame().to_csv(state_dir / "data_dictionary.csv", index=False)
coverage = pd.DataFrame({"first": daily.apply(lambda c: c.first_valid_index()), "last": daily.apply(lambda c: c.last_valid_index()),
                         "missing": daily.isna().sum()})
coverage.to_csv(qdir / "coverage.csv")
print("Saved:", paths.rel(paths.processed / "daily_panel.parquet"))
coverage